# Lakebase check (1/2): synced table, row-level parity, lookup latency, evidence decisions

Runs inside the workspace (serverless), as the app will, against Lakebase project `bigbox-ops`,
database `bigbox_ops`:
1. status and last sync of `serving.recommendations_synced` (TRIGGERED from `gold.recommendations`);
2. source vs synced, row by row: every store x SKU, the columns the app shows;
3. p50 / p95 / p99 latency of the app's two reads, timed in a loop on one connection;
4. the evidence decisions: the 20 top-priority queue rows get a decision through the same
   `INSERT ... RETURNING` the app will use, `decided_by = 'stage-d-evidence'`. A re-run for the
   same recommendation snapshot reuses them instead of inserting more.

The ids go to the `verify` task (task values), which runs after the `write_back` task.

In [ ]:
import json
import random
import statistics
import time

import pandas as pd
from databricks.sdk import WorkspaceClient
from pyspark.sql import functions as F

from bigbox_lakebase import connect

dbutils.widgets.text("catalog", "bigbox_fe_bar")
dbutils.widgets.text("n_decisions", "20")
CAT = dbutils.widgets.get("catalog")
N_DECISIONS = int(dbutils.widgets.get("n_decisions"))
SYNCED = f"{CAT}.serving.recommendations_synced"
w = WorkspaceClient()
conn = connect(w)
print("connected:", conn.execute("SELECT current_database(), current_user, version()").fetchone())

connected: ('bigbox_ops', 'ritesh.patel@databricks.com', 'PostgreSQL 17.12 (4d8000a) on aarch64-unknown-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit')


## 1. Synced table status and last sync

In [ ]:
def iso(x) -> str | None:
    """SDK timestamps are protobuf Timestamps; print them as RFC 3339."""
    if x is None:
        return None
    return x.ToJsonString() if hasattr(x, "ToJsonString") else str(x)


st = w.postgres.get_synced_table(f"synced_tables/{SYNCED}")
s = st.status
info = s.last_sync.delta_table_sync_info if s.last_sync else None
source_version = spark.sql(f"DESCRIBE HISTORY {CAT}.gold.recommendations LIMIT 1").first()[
    "version"
]
status = {
    "synced_table": SYNCED,
    "scheduling_policy": st.spec.scheduling_policy.value if st.spec else None,
    "detailed_state": s.detailed_state.value if s.detailed_state else None,
    "pipeline_id": s.pipeline_id,
    "last_sync_start": iso(s.last_sync.sync_start_time) if s.last_sync else None,
    "last_sync_end": iso(s.last_sync.sync_end_time) if s.last_sync else None,
    "synced_source_version": info.delta_commit_version if info else None,
    "synced_source_commit_time": iso(info.delta_commit_time) if info else None,
    "source_latest_version": source_version,
}
print(json.dumps(status, indent=2, default=str))

{
  "synced_table": "bigbox_fe_bar.serving.recommendations_synced",
  "scheduling_policy": null,
  "detailed_state": "SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE",
  "pipeline_id": "e1d1b737-4300-4602-80c8-dabb0e38df80",
  "last_sync_start": "2026-11-25T03:14:23.608697Z",
  "last_sync_end": "2026-11-25T03:14:26.523071Z",
  "synced_source_version": 4,
  "synced_source_commit_time": "2026-11-25T03:13:28Z",
  "source_latest_version": 4
}


## 2. Source vs synced, row by row

Rows, then every (store_id, sku_id) compared on the columns the queue and the what-if use.
Doubles are compared within 1e-9; timestamps as UTC instants.

In [ ]:
COLS = [
    "on_hand",
    "on_order_units",
    "forecast_today",
    "demand_until_delivery",
    "stockout_risk",
    "expected_lost_sales",
    "recommended_order_qty",
    "queue_reason",
    "priority",
    "as_of_ts",
    "features_json",
    "model_version",
]
src = (
    spark.read.table(f"{CAT}.gold.recommendations")
    .select("store_id", "sku_id", *COLS)
    .withColumn("as_of_ts", F.date_format("as_of_ts", "yyyy-MM-dd'T'HH:mm:ss"))
    .toPandas()
)
cur = conn.execute(
    "SELECT store_id, sku_id, " + ", ".join(COLS) + " FROM serving.recommendations_synced"
)
dst = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
dst["as_of_ts"] = pd.to_datetime(dst["as_of_ts"], utc=True).dt.strftime("%Y-%m-%dT%H:%M:%S")
m = src.merge(dst, on=["store_id", "sku_id"], how="outer", suffixes=("_src", "_pg"), indicator=True)
diff = {}
for c in COLS:
    a, b = m[f"{c}_src"], m[f"{c}_pg"]
    if pd.api.types.is_float_dtype(a) and pd.api.types.is_float_dtype(b):
        same = ((a - b).abs() <= 1e-9) | (a.isna() & b.isna())
    else:
        same = (a.astype(str) == b.astype(str)) | (a.isna() & b.isna())
    diff[c] = int((~same & (m["_merge"] == "both")).sum())
parity = {
    "source_rows": len(src),
    "synced_rows": len(dst),
    "only_in_source": int((m["_merge"] == "left_only").sum()),
    "only_in_synced": int((m["_merge"] == "right_only").sum()),
    "queue_rows_source": int(src["queue_reason"].notna().sum()),
    "queue_rows_synced": int(dst["queue_reason"].notna().sum()),
    "rows_differing_by_column": diff,
}
print(json.dumps(parity, indent=2))
assert parity["only_in_source"] == parity["only_in_synced"] == 0 and not any(diff.values())

{
  "source_rows": 28005,
  "synced_rows": 28005,
  "only_in_source": 0,
  "only_in_synced": 0,
  "queue_rows_source": 5133,
  "queue_rows_synced": 5133,
  "rows_differing_by_column": {
    "on_hand": 0,
    "on_order_units": 0,
    "forecast_today": 0,
    "demand_until_delivery": 0,
    "stockout_risk": 0,
    "expected_lost_sales": 0,
    "recommended_order_qty": 0,
    "queue_reason": 0,
    "priority": 0,
    "as_of_ts": 0,
    "features_json": 0,
    "model_version": 0
  }
}


## 3. Lookup latency (the app's two reads)

One connection, as the app's pool keeps one open. After 10 warm-up queries: 500 point lookups of
a random store x SKU (the queue item's detail and feature row, primary key) and 200 reads of a
random store's top-25 queue (partial index). Times are client-side, round trip included.

In [ ]:
keys = list(src[["store_id", "sku_id"]].itertuples(index=False, name=None))
stores = sorted(src["store_id"].unique())
rng = random.Random(7)
LOOKUP = "SELECT * FROM serving.recommendations_synced WHERE store_id = %s AND sku_id = %s"
QUEUE = (
    "SELECT store_id, sku_id, category, on_hand, stockout_risk, expected_lost_sales,"
    " recommended_order_qty, queue_reason, priority FROM serving.recommendations_synced"
    " WHERE store_id = %s AND queue_reason IS NOT NULL ORDER BY priority LIMIT 25"
)


def timed(sql, params):
    t = time.perf_counter()
    conn.execute(sql, params).fetchall()
    return (time.perf_counter() - t) * 1000


for _ in range(10):
    timed(LOOKUP, rng.choice(keys))
lat = {
    "point lookup (PK)": [timed(LOOKUP, rng.choice(keys)) for _ in range(500)],
    "store queue, top 25": [timed(QUEUE, (rng.choice(stores),)) for _ in range(200)],
}


def pct(xs, q):
    return statistics.quantiles(xs, n=100, method="inclusive")[q - 1]


latency = {
    k: {
        "n": len(v),
        "p50_ms": round(pct(v, 50), 2),
        "p95_ms": round(pct(v, 95), 2),
        "p99_ms": round(pct(v, 99), 2),
        "max_ms": round(max(v), 2),
    }
    for k, v in lat.items()
}
plan = conn.execute("EXPLAIN " + QUEUE, (stores[0],)).fetchall()
print(json.dumps(latency, indent=2))
print("queue plan:", plan[0][0], "|", " ".join(r[0].strip() for r in plan[1:3]))

{
  "point lookup (PK)": {
    "n": 500,
    "p50_ms": 2.2,
    "p95_ms": 3.88,
    "p99_ms": 4.91,
    "max_ms": 20.37
  },
  "store queue, top 25": {
    "n": 200,
    "p50_ms": 2.31,
    "p95_ms": 4.23,
    "p99_ms": 5.95,
    "max_ms": 10.16
  }
}
queue plan: Limit  (cost=0.28..50.70 rows=25 width=68) | ->  Index Scan using partition_17057_store_id_priority_idx on partition_17057 recommendations_synced  (cost=0.28..99.10 rows=49 width=68) Index Cond: (store_id = 'S001'::text)


## 4. Evidence decisions into `app.reorder_decisions`

The 20 top-priority queue rows of the current snapshot, actions in turn approve / override /
reject. Override adds one case (reason: planner judgement); reject orders nothing. This is the
INSERT the app runs, so the CHECK constraints apply.

In [ ]:
INSERT = """
INSERT INTO app.reorder_decisions
  (store_id, sku_id, rec_as_of_ts, recommended_qty, final_qty, action, reason, decided_by)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
RETURNING decision_id
"""
EVIDENCE_USER = "stage-d-evidence"
top = conn.execute(
    "SELECT store_id, sku_id, as_of_ts, recommended_order_qty, case_pack"
    " FROM serving.recommendations_synced WHERE queue_reason IS NOT NULL"
    " ORDER BY priority LIMIT %s",
    (N_DECISIONS,),
).fetchall()
snapshot = top[0][2]
existing = conn.execute(
    "SELECT decision_id FROM app.reorder_decisions WHERE decided_by = %s AND rec_as_of_ts = %s"
    " ORDER BY decided_at",
    (EVIDENCE_USER, snapshot),
).fetchall()
before = conn.execute("SELECT count(*) FROM app.reorder_decisions").fetchone()[0]
if existing:
    ids = [str(r[0]) for r in existing]
    print(f"{len(ids)} evidence decisions already exist for snapshot {snapshot}; reusing them")
else:
    ids = []
    with conn.transaction():
        for i, (store, sku, as_of, qty, case) in enumerate(top):
            action = ("approve", "override", "reject")[i % 3]
            final, reason = {
                "approve": (qty, None),
                "override": (qty + case, "Planner judgement: one extra case (stage-D evidence)"),
                "reject": (0, "Planner judgement: no order tonight (stage-D evidence)"),
            }[action]
            row = conn.execute(
                INSERT, (store, sku, as_of, qty, final, action, reason, EVIDENCE_USER)
            ).fetchone()
            ids.append(str(row[0]))
after = conn.execute("SELECT count(*) FROM app.reorder_decisions").fetchone()[0]
by_action = dict(
    conn.execute(
        "SELECT action, count(*) FROM app.reorder_decisions WHERE decision_id = ANY(%s::uuid[])"
        " GROUP BY action ORDER BY action",
        (ids,),
    ).fetchall()
)
seed = {
    "snapshot_as_of_ts": snapshot,
    "postgres_rows_before": before,
    "postgres_rows_after": after,
    "evidence_decisions": len(ids),
    "by_action": by_action,
    "first_ids": ids[:3],
}
print(json.dumps(seed, indent=2, default=str))
dbutils.jobs.taskValues.set("decision_ids", ids)
conn.close()
summary = {"status": status, "parity": parity, "latency": latency, "seed": seed}
dbutils.notebook.exit(json.dumps(summary, default=str))  # read by scripts/evidence_lakebase.py

{"status": {"synced_table": "bigbox_fe_bar.serving.recommendations_synced", "scheduling_policy": null, "detailed_state": "SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE", "pipeline_id": "e1d1b737-4300-4602-80c8-dabb0e38df80", "last_sync_start": "2026-11-25T03:14:23.608697Z", "last_sync_end": "2026-11-25T03:14:26.523071Z", "synced_source_version": 4, "synced_source_commit_time": "2026-11-25T03:13:28Z", "source_latest_version": 4}, "parity": {"source_rows": 28005, "synced_rows": 28005, "only_in_source": 0, "only_in_synced": 0, "queue_rows_source": 5133, "queue_rows_synced": 5133, "rows_differing_by_column": {"on_hand": 0, "on_order_units": 0, "forecast_today": 0, "demand_until_delivery": 0, "stockout_risk": 0, "expected_lost_sales": 0, "recommended_order_qty": 0, "queue_reason": 0, "priority": 0, "as_of_ts": 0, "features_json": 0, "model_version": 0}}, "latency": {"point lookup (PK)": {"n": 500, "p50_ms": 2.2, "p95_ms": 3.88, "p99_ms": 4.91, "max_ms": 20.37}, "store queue, top 25": {"n": 200, "p5